# CFPB Lender x Sub-product Feature Matrix

**Unit of analysis:** `canonical_company` x `Sub-product` -> 2,415 groups.

**Purpose:** the clean quantitative feature table that will eventually feed the FinePrint
Safety Label.

**Scope discipline.** This notebook assigns no weights, computes no score, runs no
clustering or dimensionality reduction, and uses no narrative text. Response rates are
recorded as observed rates with no good/bad interpretation attached. Peer differences are
signed percentage-point deviations and nothing more.

**Inputs:** `data/raw/paydayComplaints.csv` (read-only) and
`data/processed/company_aliases.csv` from the company-identity work.

In [1]:
import itertools
import os
import re
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

REPO = "/Users/mihirkale/cdc"
RAW = f"{REPO}/data/raw/paydayComplaints.csv"
ALIASES = f"{REPO}/data/processed/company_aliases.csv"
OUT = f"{REPO}/data/processed/lender_subproduct_features.csv"

PAYDAY = "Payday loan"
MIN_N = 20
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)

## 0. Load and apply the canonical mapping

In [2]:
raw = pd.read_csv(RAW)
aliases = pd.read_csv(ALIASES)

print(f"raw complaints       : {len(raw):,}")
print(f"alias table rows     : {len(aliases):,}")
print(f"every Company mapped : {set(raw['Company']) <= set(aliases['original_company'])}")

df = raw.copy()
df["canonical_company"] = df["Company"].map(dict(zip(aliases["original_company"],
                                                     aliases["canonical_company"])))
df["canonical_key"] = df["Company"].map(dict(zip(aliases["original_company"],
                                                 aliases["canonical_key"])))

print(f"\ncomplaints after mapping : {len(df):,}  (conserved: {len(df) == len(raw)})")
print(f"distinct Company values  : {raw['Company'].nunique():,} -> {df['canonical_company'].nunique():,}")
print(f"nulls in key columns     : {df[['canonical_company','canonical_key']].isna().sum().sum()}")

raw complaints       : 38,375
alias table rows     : 1,162
every Company mapped : True

complaints after mapping : 38,375  (conserved: True)
distinct Company values  : 1,162 -> 1,161
nulls in key columns     : 0


In [3]:
groups = df.groupby(["canonical_company", "Sub-product"]).size()
print(f"lender x sub-product groups : {len(groups):,}")
print(f"complaints conserved        : {int(groups.sum()) == len(df)}")
print(f"unique group keys           : {df.groupby(['canonical_company','Sub-product']).ngroups:,}")
print(f"median group size           : {groups.median():.0f}   mean {groups.mean():.1f}")
print(f"canonical companies present : {df['canonical_company'].nunique():,}")

lender x sub-product groups : 2,415
complaints conserved        : True
unique group keys           : 2,415
median group size           : 2   mean 15.9
canonical companies present : 1,161


## 1. Issue features and a label-quality check

`n_ij` = complaints of Issue *j* for lender-product *i*; `p_ij = n_ij / n_i`.
Issues are numbered `i01`..`i33` in descending national frequency, alphabetical tie-break,
so column order is meaningful and reproducible.

Before building columns, the 33 raw labels are checked for near-duplicates. This matters
because a silent duplicate splits a real signal in half across two columns.

In [4]:
issues = (raw["Issue"].value_counts().rename_axis("issue").reset_index(name="n")
              .sort_values(["n", "issue"], ascending=[False, True]))
issues["key"] = [f"i{i:02d}" for i in range(1, len(issues) + 1)]
issues["slug"] = [re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")[:34].rstrip("_")
                  for s in issues["issue"]]
print(f"{len(issues)} distinct Issues, {raw['Issue'].isna().sum()} nulls")
issues.head(8)[["key", "n", "issue"]].to_string(index=False)

33 distinct Issues, 0 nulls


"key    n                                                  issue\ni01 9225             Charged fees or interest you didn't expect\ni02 5349                           Problem when making payments\ni03 4145                            Struggling to pay your loan\ni04 3401                                       Getting the loan\ni05 2967    Problem with additional add-on products or services\ni06 2887 Problem with the payoff process at the end of the loan\ni07 2596                   Incorrect information on your report\ni08 1258                               Getting a line of credit"

In [5]:
norm = {l: re.sub(r"[^a-z0-9]+", " ", l.lower()).strip() for l in issues["issue"]}
pairs = [(SequenceMatcher(None, norm[a], norm[b]).ratio(), a, b)
         for a, b in itertools.combinations(issues["issue"], 2)]
near = [(r, a, b) for r, a, b in pairs if r >= 0.85]
print("=== near-duplicate Issue labels (difflib >= 0.85 on normalised text) ===")
for r, a, b in near:
    ca, cb = int(issues.loc[issues.issue == a, "n"].iloc[0]), int(issues.loc[issues.issue == b, "n"].iloc[0])
    print(f"  {r:.2f}  {ca:>5,}  {a!r}")
    print(f"         {cb:>5,}  {b!r}")
print(f"  pairs flagged: {len(near)}")

=== near-duplicate Issue labels (difflib >= 0.85 on normalised text) ===
  0.96    258  "Was approved for a loan, but didn't receive the money"
            61  "Was approved for a loan, but didn't receive money"
  pairs flagged: 1


In [6]:
print("=== why the obvious merges are NOT made ===")
for a, b in [("Charged fees or interest you didn't expect", "Unexpected fees"),
             ("Vehicle was repossessed or sold the vehicle", "Vehicle was damaged or destroyed the vehicle"),
             ("Property was sold", "Property was damaged or destroyed property")]:
    na = int(issues.loc[issues.issue == a, "n"].iloc[0])
    nb = int(issues.loc[issues.issue == b, "n"].iloc[0])
    sa = int(raw.loc[raw.Issue == a, "Sub-issue"].notna().sum())
    sb = int(raw.loc[raw.Issue == b, "Sub-issue"].notna().sum())
    print(f"  {na:>5,}  {a!r}   sub-issue populated {sa}")
    print(f"  {nb:>5,}  {b!r}   sub-issue populated {sb}   ratio {SequenceMatcher(None,norm[a],norm[b]).ratio():.2f}")
print("\n  These pairs are semantically distinct even where the wording overlaps, so all 33")
print("  columns are kept separate. The one true near-duplicate (i18 / i28) is flagged, not")
print("  merged, so the 33-column contract is preserved and the defect stays visible.")

=== why the obvious merges are NOT made ===
  9,225  "Charged fees or interest you didn't expect"   sub-issue populated 0
    173  'Unexpected fees'   sub-issue populated 0   ratio 0.28
    376  'Vehicle was repossessed or sold the vehicle'   sub-issue populated 0
     94  'Vehicle was damaged or destroyed the vehicle'   sub-issue populated 0   ratio 0.76
     18  'Property was sold'   sub-issue populated 0
      6  'Property was damaged or destroyed property'   sub-issue populated 0   ratio 0.54

  These pairs are semantically distinct even where the wording overlaps, so all 33
  columns are kept separate. The one true near-duplicate (i18 / i28) is flagged, not
  merged, so the 33-column contract is preserved and the defect stays visible.


In [7]:
# Full cross product of groups x Issues, so an absent Issue is an explicit 0, never NaN.
gkeys = (df.groupby(["canonical_company", "Sub-product"]).size()
           .rename("n_complaints").reset_index())
pair = (df.groupby(["canonical_company", "Sub-product", "Issue"]).size().rename("n")
          .reset_index().rename(columns={"Issue": "issue"}))

grid = (gkeys.merge(issues[["key", "issue"]], how="cross")
            .merge(pair, on=["canonical_company", "Sub-product", "issue"], how="left"))
print(f"grid rows {len(grid):,} = {len(gkeys):,} groups x {len(issues)} issues "
      f"(expected {len(gkeys) * len(issues):,})")
grid["n"] = grid["n"].fillna(0.0)
grid["p"] = grid["n"] / grid["n_complaints"]
print(f"NaNs anywhere in grid            : {int(grid.isna().sum().sum())}")
print(f"counts reconcile to group totals : "
      f"{np.allclose(grid.groupby(['canonical_company','Sub-product'])['n'].sum().values, groups.values)}")
print(f"share rows summing to exactly 1.0: "
      f"{int((grid.groupby(['canonical_company','Sub-product'])['p'].sum() == 1.0).sum()):,} of {len(groups):,}")

grid rows 79,695 = 2,415 groups x 33 issues (expected 79,695)
NaNs anywhere in grid            : 0
counts reconcile to group totals : True
share rows summing to exactly 1.0: 2,354 of 2,415


## 2. Response features

Five rates plus the counts needed to audit them. Recorded, not interpreted.

One thing to stay careful about: **two different things are called "untimely" in this
dataset and they are not the same number.** `Company response to consumer =
"Untimely response"` flags 535 complaints, whereas `Timely response? = "No"` covers 1,732,
because 1,157 complaints were answered late and still closed as "Closed with explanation".
Both are kept so the distinction survives into the feature table.

In [8]:
RESP = "Company response to consumer"
print("=== exact category values (repr, so misspellings cannot hide) ===")
for v, n in raw[RESP].value_counts().items():
    print(f"  {n:>6,}  {v!r}")
print()
for v, n in raw["Timely response?"].value_counts().items():
    print(f"  {n:>6,}  {v!r}")
print(f"\n  nulls: response={raw[RESP].isna().sum()}  timely={raw['Timely response?'].isna().sum()}")

=== exact category values (repr, so misspellings cannot hide) ===
  34,598  'Closed with explanation'
   1,423  'Closed with non-monetary relief'
   1,261  'Closed with monetary relief'
     558  'In progress'
     535  'Untimely response'

  36,643  'Yes'
   1,732  'No'

  nulls: response=0  timely=0


In [9]:
resp = (df.groupby(["canonical_company", "Sub-product"])
          .agg(n_complaints=("Complaint ID", "size"),
               n_timely_yes=("Timely response?", lambda s: int((s == "Yes").sum())),
               n_timely_no=("Timely response?", lambda s: int((s == "No").sum())),
               n_monetary_relief=("Company response to consumer",
                                  lambda s: int((s == "Closed with monetary relief").sum())),
               n_nonmonetary_relief=("Company response to consumer",
                                     lambda s: int((s == "Closed with non-monetary relief").sum())),
               n_untimely_response=("Company response to consumer",
                                    lambda s: int((s == "Untimely response").sum())),
               n_in_progress=("Company response to consumer",
                              lambda s: int((s == "In progress").sum())),
               n_public_response_null=("Company public response", lambda s: int(s.isna().sum())),
               n_states=("State", "nunique"),
               date_first=("Date received", "min"),
               date_last=("Date received", "max"))
          .reset_index())

RATE_SRC = {"timely_response": "n_timely_yes",
            "not_timely": "n_timely_no",
            "monetary_relief": "n_monetary_relief",
            "nonmonetary_relief": "n_nonmonetary_relief",
            "untimely_response": "n_untimely_response",
            "in_progress": "n_in_progress",
            "public_response_null": "n_public_response_null"}
for num, src in RATE_SRC.items():
    resp[f"rate__{num}"] = resp[src] / resp["n_complaints"]

print(f"groups: {len(resp):,}")
resp[["canonical_company", "Sub-product", "n_complaints", "n_timely_yes",
      "n_untimely_response", "n_in_progress", "rate__timely_response"]].head(5)

groups: 2,415


,canonical_company,Sub-product,n_complaints,n_timely_yes,n_untimely_response,n_in_progress,rate__timely_response
0,"1 STOP MONEY CENTERS, LLC",Personal line of credit,1,0,1,0,0.0
1,1 Stop Payday Loan Services LLC,Payday loan,1,0,1,0,0.0
2,1st Franklin Financial Corporation,Installment loan,70,70,0,0,1.0
3,1st Franklin Financial Corporation,Payday loan,4,4,0,0,1.0
4,1st Franklin Financial Corporation,Personal line of credit,18,18,0,0,1.0


## 3. Peer baselines and deltas

For each `Sub-product` the peer baseline for Issue *j* is the **complaint-weighted** pooled
share across all lenders filing in that sub-product:

```
p_peer(j,s) = sum_i n_ij / sum_i n_i        over all i in sub-product s
d_ij        = 100 * (p_ij - p_peer(j,s))
```

The baseline includes lender *i* itself. Across 482 payday lenders that self-influence is
negligible in aggregate, but it is not exactly zero for an individual small lender; a
leave-one-out baseline would be the cleaner choice if these deltas ever become load-bearing.

In [10]:
peer = (grid.groupby(["Sub-product", "key", "issue"], as_index=False)["n"].sum()
            .merge(grid.groupby("Sub-product")["n"].sum().rename("sub_n"),
                   on="Sub-product"))
peer["peer_baseline_pct"] = (peer["n"] / peer["sub_n"] * 100).round(4)

# The denominator is the total complaint count of the sub-product. Summing `n_complaints`
# here instead would multiply it by the 33 grid rows per group and shrink every baseline.
chk = grid.groupby("Sub-product")["n"].sum().rename("via_counts").reset_index()
chk["direct"] = df.groupby("Sub-product").size().values
print(f"peer denominator equals raw sub-product totals: {(chk['via_counts'] == chk['direct']).all()}")
print("\npeer baselines, Payday loan (top 8 by volume):")
print(peer[peer["Sub-product"] == PAYDAY].sort_values("n", ascending=False)
          .head(8)[["key", "issue", "n", "peer_baseline_pct"]].to_string(index=False))

peer denominator equals raw sub-product totals: True

peer baselines, Payday loan (top 8 by volume):
key                                                                           issue      n  peer_baseline_pct
i01                                      Charged fees or interest you didn't expect 1783.0            29.5983
i03                                                     Struggling to pay your loan 1066.0            17.6959
i11                                   Can't stop withdrawals from your bank account  605.0            10.0432
i10                                            Received a loan you didn't apply for  586.0             9.7278
i12                                                Can't contact lender or servicer  391.0             6.4907
i06                          Problem with the payoff process at the end of the loan  373.0             6.1919
i17 Money was taken from your bank account on the wrong day or for the wrong amount  316.0             5.2457
i16                

In [11]:
# Validation against the worked example quoted in the task brief.
fee_key = issues.loc[issues.issue == "Charged fees or interest you didn't expect", "key"].iloc[0]
base = peer[(peer["Sub-product"] == PAYDAY) & (peer["key"] == fee_key)]["peer_baseline_pct"].iloc[0]
print(f"payday peer baseline, unexpected fees : {base:.2f}%   (brief says 29.6%)")

# Restrict the illustration to groups with enough complaints: a 1-complaint group whose only
# complaint is a fee complaint has p=1.0 and would otherwise top this table as a "96.7 pp
# outlier", which is a sample-size artefact rather than lender behaviour.
d = grid[(grid["Sub-product"] == PAYDAY) & (grid["key"] == fee_key)
         & (grid["n_complaints"] >= MIN_N)].merge(
    peer[peer["Sub-product"] == PAYDAY][["key", "peer_baseline_pct"]], on="key")
d["delta_pp"] = (d["p"] * 100 - d["peer_baseline_pct"]).round(1)
print(f"\nlargest positive deviations, payday lenders with >={MIN_N} complaints "
      f"({len(d)} groups):")
print(d.nlargest(3, "delta_pp")[["canonical_company", "n", "n_complaints", "p",
                                 "peer_baseline_pct", "delta_pp"]]
        .to_string(index=False, float_format=lambda x: f"{x:7.3f}"))
print("\nNote: 29.60% is the baseline over ALL 6,024 payday complaints. Section 6 reports a "
      "31.7% baseline\nfor the >=20-complaint subset, which is a different, smaller "
      "denominator by construction.")

payday peer baseline, unexpected fees : 29.60%   (brief says 29.6%)

largest positive deviations, payday lenders with >=20 complaints (70 groups):
                      canonical_company       n  n_complaints       p  peer_baseline_pct  delta_pp
Benht: Economic Development Corporation  38.000            47   0.809             29.598    51.300
                     Deer Creek Lending  24.000            34   0.706             29.598    41.000
                 Post Lake Lending Inc.  30.000            44   0.682             29.598    38.600

Note: 29.60% is the baseline over ALL 6,024 payday complaints. Section 6 reports a 31.7% baseline
for the >=20-complaint subset, which is a different, smaller denominator by construction.


## 4. Assemble the feature table

In [12]:
base_cols = (df.groupby(["canonical_company", "canonical_key", "Sub-product"])
               .size().rename("n_complaints").reset_index())

cnt = grid.pivot_table(index=["canonical_company", "Sub-product"], columns="key",
                       values="n", aggfunc="sum")
shr = grid.pivot_table(index=["canonical_company", "Sub-product"], columns="key",
                       values="p", aggfunc="sum")

dlt_parts = []
for s, gp in grid.groupby("Sub-product"):
    b = peer[peer["Sub-product"] == s].set_index("key")["peer_baseline_pct"]
    dlt_parts.append(gp.pivot_table(index=["canonical_company", "Sub-product"], columns="key",
                                    values="p", aggfunc="sum").mul(100).sub(b, axis=1))
dlt = pd.concat(dlt_parts)

feat = (base_cols
        .merge(resp.drop(columns=["n_complaints"]), on=["canonical_company", "Sub-product"])
        .merge(cnt.add_prefix("cnt__").round(0), on=["canonical_company", "Sub-product"])
        .merge(shr.add_prefix("shr__").round(6), on=["canonical_company", "Sub-product"])
        .merge(dlt.add_prefix("dlt__").round(4), on=["canonical_company", "Sub-product"]))

feat = feat.sort_values(["Sub-product", "n_complaints"], ascending=[True, False]).reset_index(drop=True)
print(f"rows   : {len(feat):,}  (expected {len(groups):,})")
print(f"columns: {feat.shape[1]}")
print(f"groups lost in assembly: {len(groups) - len(feat)}")
feat[["canonical_company", "Sub-product", "n_complaints",
      "rate__timely_response", "rate__monetary_relief"]].head(5)

rows   : 2,415  (expected 2,415)
columns: 120
groups lost in assembly: 0


,canonical_company,Sub-product,n_complaints,rate__timely_response,rate__monetary_relief
0,"DailyPay, Inc",Earned wage access,48,1.000000,0.020833
1,Activehours,Earned wage access,44,0.977273,0.181818
2,"Dave Operating, LLC",Earned wage access,39,0.974359,0.000000
3,"Payactiv, Inc.",Earned wage access,28,0.750000,0.000000
4,Albert Corporation,Earned wage access,15,1.000000,0.000000


In [13]:
keys = issues["key"].tolist()
checks = {
    "complaints sum to 38,375": int(feat["n_complaints"].sum()) == len(raw),
    "one row per group": not feat.duplicated(["canonical_company", "Sub-product"]).any(),
    "issue counts reconcile": all(
        np.allclose(feat[f"cnt__{k}"].sum(), grid[grid["key"] == k]["n"].sum()) for k in keys),
    "shares sum to 1 per group": bool(np.allclose(
        feat[[f"shr__{k}" for k in keys]].sum(axis=1), 1.0, atol=1e-4)),
    "complaint-weighted deltas average ~0 per sub-product": bool(
        (feat[[f"dlt__{k}" for k in keys]].mul(feat["n_complaints"], axis=0)
             .groupby(feat["Sub-product"]).sum()
             .div(feat["n_complaints"].groupby(feat["Sub-product"]).sum(), axis=0)
             .abs().to_numpy().max() < 1e-2)),
    "rates in [0,1]": bool(all(feat[c].between(0, 1).all()
                               for c in feat.columns if c.startswith("rate__"))),
    "no NaN in any count/share/delta": bool(
        not feat[[f"{p}{k}" for p in ("cnt__", "shr__", "dlt__") for k in keys]].isna().any().any()),
    "every group has explicit zero counts": bool(
        (feat[[f"cnt__{k}" for k in keys]] == 0).any(axis=1).all()),
    "33 issue cols per family": all(
        len([c for c in feat.columns if c.startswith(p)]) == 33 for p in ("cnt__", "shr__", "dlt__")),
    "no n_complaints drift": bool(np.array_equal(
        feat.sort_values(["canonical_company", "Sub-product"])["n_complaints"].values,
        groups.sort_index().values)),
}
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print(f"\nall checks pass: {all(checks.values())}")

  PASS  complaints sum to 38,375
  PASS  one row per group
  PASS  issue counts reconcile
  PASS  shares sum to 1 per group
  PASS  complaint-weighted deltas average ~0 per sub-product
  PASS  rates in [0,1]
  PASS  no NaN in any count/share/delta
  PASS  every group has explicit zero counts
  PASS  33 issue cols per family
  PASS  no n_complaints drift

all checks pass: True


In [14]:
tmp = f"{OUT}.tmp"
feat.to_csv(tmp, index=False)
os.replace(tmp, OUT)
print(f"wrote {OUT}  ({os.path.getsize(OUT):,} bytes)")
chk = pd.read_csv(OUT)
print(f"round-trip rows={len(chk):,} cols={chk.shape[1]}")
print("\ncolumn families:")
for pre, desc in [("cnt__", "issue counts"), ("shr__", "issue shares"),
                  ("dlt__", "peer deltas (pp)"), ("rate__", "response rates"),
                  ("n_", "counts / context")]:
    print(f"  {pre:>8}  {len([c for c in chk.columns if c.startswith(pre)]):>4}  {desc}")

wrote /Users/mihirkale/cdc/data/processed/lender_subproduct_features.csv  (1,501,793 bytes)
round-trip rows=2,415 cols=120

column families:
     cnt__    33  issue counts
     shr__    33  issue shares
     dlt__    33  peer deltas (pp)
    rate__     7  response rates
        n_     9  counts / context


## 5. Sample-size coverage

In [15]:
print("=== overall, by minimum complaint count per group ===")
rows = []
for t in (1, 5, 10, 20, 30, 50, 100):
    sel = groups >= t
    rows.append({"min_n": t, "groups": int(sel.sum()),
                 "pct_of_groups": round(sel.mean() * 100, 1),
                 "pct_of_complaints": round(groups[sel].sum() / groups.sum() * 100, 1)})
cov = pd.DataFrame(rows)
cov

=== overall, by minimum complaint count per group ===


,min_n,groups,pct_of_groups,pct_of_complaints
0,1,2415,100.0,100.0
1,5,706,29.2,92.8
2,10,458,19.0,88.5
3,20,316,13.1,83.3
4,30,239,9.9,78.5
5,50,163,6.7,70.9
6,100,82,3.4,56.6


In [16]:
print("=== per Sub-product ===")
rows = []
for s, d in df.groupby("Sub-product"):
    g = d.groupby("canonical_company").size()
    rows.append({"Sub-product": s, "complaints": len(d), "groups": len(g),
                 "median_n": g.median(),
                 **{f"n>={t}": int((g >= t).sum()) for t in (5, 10, 20, 30, 50, 100)}})
cov_sp = pd.DataFrame(rows).sort_values("complaints", ascending=False)
cov_sp

=== per Sub-product ===


,Sub-product,complaints,groups,median_n,n>=5,n>=10,n>=20,n>=30,n>=50,n>=100
1,Installment loan,21835,804,2.0,275,194,149,122,95,52
5,Personal line of credit,6712,507,2.0,171,113,76,57,30,12
4,Payday loan,6024,482,2.0,151,100,70,48,32,14
7,Title loan,2488,210,1.0,48,24,11,9,6,4
2,Other advances of future income,731,235,1.0,40,14,5,0,0,0
0,Earned wage access,396,93,2.0,15,9,4,3,0,0
3,Pawn loan,127,53,1.0,4,3,1,0,0,0
6,Tax refund anticipation loan or check,62,31,1.0,2,1,0,0,0,0


In [17]:
print("=== is the largest Issue the same in every sub-product? ===")
t = df.groupby(["Sub-product", "Issue"]).size().rename("n").reset_index()
t["rank"] = t.groupby("Sub-product")["n"].rank(ascending=False, method="first").astype(int)
top = t[t["rank"] == 1][["Sub-product", "Issue", "n"]].sort_values("n", ascending=False)
top["is_unexpected_fees"] = top["Issue"] == "Charged fees or interest you didn't expect"
print(top.to_string(index=False))
print(f"\n  unexpected-fees is #1 in {int(top['is_unexpected_fees'].sum())} of {len(top)} sub-products")

=== is the largest Issue the same in every sub-product? ===
                          Sub-product                                            Issue    n  is_unexpected_fees
                     Installment loan       Charged fees or interest you didn't expect 5206                True
                          Payday loan       Charged fees or interest you didn't expect 1783                True
              Personal line of credit       Charged fees or interest you didn't expect 1493                True
                           Title loan       Charged fees or interest you didn't expect  681                True
      Other advances of future income                            Issues with repayment  201               False
                   Earned wage access                            Issues with repayment  169               False
                            Pawn loan       Charged fees or interest you didn't expect   62                True
Tax refund anticipation loan or check Confus

## 6. Which Issues actually differentiate lenders

Two lenders having similar issue mixes is not evidence that an Issue carries signal: with
30 complaints one issue can move a share by 3.3 points on noise alone. So each Issue is
tested for **over-dispersion** — does the spread of `p_ij` across lenders exceed what
independent sampling would produce if every lender shared one common rate?

```
obs = weighted mean of (p_ij - mu)^2 across lenders,  weights n_i
exp = weighted mean of p(1-p)/n_i across lenders,      weights n_i
ratio = obs / exp
```

Both sides are weighted *means* over the same lenders, so `ratio` is dimensionless and
compares like with like. (Summing the variance term across lenders instead would produce a
quantity that grows with lender count and is not a variance — `ratio` would then collapse
toward zero for no real reason.) `ratio = 1` means lenders differ exactly as much as
sampling noise predicts.

In [18]:
def dispersion(shares, weights, base):
    """Observed weighted spread of shares vs the spread expected under one common rate."""
    mu = np.average(shares, weights=weights)
    obs = np.average((shares - mu) ** 2, weights=weights)
    exp = np.average(base * (1 - base) / weights, weights=weights)
    return obs, exp, (obs / exp if exp > 0 else np.nan)


NULL_HI = 1.45   # fixed decision threshold, see the band check below

print("=== calibration: simulate lenders that all share ONE true rate; ratio must sit at 1.0 ===")
rng = np.random.default_rng(7)
worst = 0.0
for base in (0.02, 0.05, 0.10, 0.20, 0.32, 0.45):
    ratios = []
    for _ in range(3000):
        n = rng.integers(20, 121, size=70).astype(float)
        k = rng.binomial(n.astype(int), base)
        ratios.append(dispersion(k / n, n, base)[2])
    ratios = np.array(ratios)
    hi = float(np.percentile(ratios, 97.5))
    worst = max(worst, hi)
    print(f"  base={base:.2f}  mean ratio={ratios.mean():.3f}  95% null band=[{np.percentile(ratios,2.5):.3f}, {hi:.3f}]")

print(f"\n  highest null upper bound across base rates: {worst:.3f}")
assert worst < NULL_HI, f"observed null band {worst:.3f} reaches the decision threshold {NULL_HI}"
print(f"  every simulated null band lies below the fixed decision threshold {NULL_HI}")
print("  -> ratios at or below the threshold are indistinguishable from sampling noise")
print("  The threshold is a fixed constant rather than the simulated maximum, so the")
print("  classification below cannot drift with Monte Carlo noise.")

=== calibration: simulate lenders that all share ONE true rate; ratio must sit at 1.0 ===
  base=0.02  mean ratio=0.984  95% null band=[0.646, 1.421]


  base=0.05  mean ratio=0.985  95% null band=[0.669, 1.380]


  base=0.10  mean ratio=0.990  95% null band=[0.680, 1.369]
  base=0.20  mean ratio=0.983  95% null band=[0.677, 1.334]


  base=0.32  mean ratio=0.987  95% null band=[0.698, 1.346]


  base=0.45  mean ratio=0.990  95% null band=[0.682, 1.335]

  highest null upper bound across base rates: 1.421
  every simulated null band lies below the fixed decision threshold 1.45
  -> ratios at or below the threshold are indistinguishable from sampling noise
  The threshold is a fixed constant rather than the simulated maximum, so the
  classification below cannot drift with Monte Carlo noise.


In [19]:
COLS_DIFF = ["key", "issue", "complaints", "lenders_any", "n_lenders", "peer_pct",
             "ratio", "spread_pp", "median_share_pct", "max_share_pct", "above_null"]


def differentiate(sub_product=PAYDAY, min_n=MIN_N):
    """Over-dispersion of each Issue's complaint share across adequately sized lenders."""
    d = df[df["Sub-product"] == sub_product]
    n = d.groupby("canonical_company").size()
    n = n[n >= min_n]
    if n.empty:
        # Several sub-products have no lender at this threshold; return a well-formed
        # empty frame rather than letting sort_values fail on a missing column.
        return pd.DataFrame(columns=COLS_DIFF)
    d = d[d["canonical_company"].isin(n.index)]
    n = d.groupby("canonical_company").size().astype(float)

    rows = []
    for key, issue in zip(issues["key"], issues["issue"]):
        c = d[d["Issue"] == issue].groupby("canonical_company").size().reindex(n.index).fillna(0.0)
        if c.sum() == 0:
            continue
        p = (c / n).to_numpy()
        base = c.sum() / n.sum()
        obs, exp, ratio = dispersion(p, n.to_numpy(), base)
        rows.append({
            "key": key, "issue": issue, "complaints": int(c.sum()),
            "lenders_any": int((c > 0).sum()), "n_lenders": len(n),
            "peer_pct": base * 100, "ratio": ratio,
            "spread_pp": float(np.percentile(p, 90) - np.percentile(p, 10)) * 100,
            "median_share_pct": float(np.median(p)) * 100,
            "max_share_pct": float(p.max()) * 100,
            "above_null": ratio > NULL_HI,
        })
    return pd.DataFrame(rows).sort_values("ratio", ascending=False).reset_index(drop=True)


pay20 = differentiate(PAYDAY, 20)
print(f"Payday lenders with >=20 complaints: {pay20['n_lenders'].iloc[0]}")
print(f"Issues occurring at all in Payday : {len(pay20)} of {len(issues)}")
pay20.to_string(index=False, float_format=lambda x: f"{x:8.2f}")

Payday lenders with >=20 complaints: 70
Issues occurring at all in Payday : 15 of 33


"key                                                                           issue  complaints  lenders_any  n_lenders  peer_pct    ratio  spread_pp  median_share_pct  max_share_pct  above_null\ni11                                   Can't stop withdrawals from your bank account         514           52         70     10.92    19.21      29.17              4.22          69.72        True\ni01                                      Charged fees or interest you didn't expect        1491           68         70     31.67    11.74      52.84             23.61          80.85        True\ni17 Money was taken from your bank account on the wrong day or for the wrong amount         267           47         70      5.67     5.50      14.22              2.01          32.26        True\ni10                                            Received a loan you didn't apply for         434           65         70      9.22     5.43      18.47              8.18          63.64        True\ni07                

### Threshold sensitivity

The statistic depends on the minimum group size, so it is recomputed at a looser threshold
and the two rankings compared.

In [20]:
pay10 = differentiate(PAYDAY, 10)
m = pay20[["key", "issue", "complaints", "ratio"]].merge(
    pay10[["key", "ratio"]], on="key", suffixes=("_n20", "_n10"))
m["rank_n20"] = m["ratio_n20"].rank(ascending=False)
m["rank_n10"] = m["ratio_n10"].rank(ascending=False)
m["rank_move"] = (m["rank_n20"] - m["rank_n10"]).abs()
print(f"max rank movement: {int(m['rank_move'].max())}   |move|>3: {int((m['rank_move'] > 3).sum())}  >5: {int((m['rank_move'] > 5).sum())}")
print("\nlargest movements:")
print(m.sort_values("rank_move", ascending=False)
       .head(5)[["issue", "complaints", "ratio_n10", "ratio_n20", "rank_n10", "rank_n20", "rank_move"]]
       .to_string(index=False, float_format=lambda x: f"{x:7.2f}"))
print(f"\ntop-6 at n>=10: {m.nsmallest(6,'rank_n10')['issue'].tolist()}")
print(f"top-6 at n>=20: {m.nsmallest(6,'rank_n20')['issue'].tolist()}")

max rank movement: 4   |move|>3: 1  >5: 0

largest movements:
                                                                          issue  complaints  ratio_n10  ratio_n20  rank_n10  rank_n20  rank_move
                          Was approved for a loan, but didn't receive the money         164       4.16       2.08      7.00     11.00       4.00
Money was taken from your bank account on the wrong day or for the wrong amount         267       4.21       5.50      5.00      3.00       2.00
                                           Incorrect information on your report         137       4.25       5.23      3.00      5.00       2.00
                               Unable to get your credit report or credit score           6       1.13       1.48     14.00     12.00       2.00
                         Problem with the payoff process at the end of the loan         284       1.34       1.34     12.00     14.00       2.00

top-6 at n>=10: ["Can't stop withdrawals from your bank account", "

### Issues showing no lender heterogeneity, and thin Issues

In [21]:
print(f"=== Issues with ratio <= null band ({NULL_HI:.2f}): no evidence of lender heterogeneity ===")
flat = pay20[~pay20["above_null"]]
print(f"count: {len(flat)} of {len(pay20)}")
print(flat[["issue", "complaints", "lenders_any", "peer_pct", "ratio", "spread_pp"]]
        .to_string(index=False, float_format=lambda x: f"{x:7.2f}"))

=== Issues with ratio <= null band (1.45): no evidence of lender heterogeneity ===
count: 3 of 15
                                                  issue  complaints  lenders_any  peer_pct   ratio  spread_pp
          Problem with fraud alerts or security freezes          13           10      0.28    1.42       0.98
 Problem with the payoff process at the end of the loan         284           64      6.03    1.34       8.64
Credit monitoring or identity theft protection services          15           12      0.32    0.89       1.23


In [22]:
print("=== Issues dominated by small samples (Payday) ===")
thin = pay20[pay20["complaints"] < 30].sort_values("complaints")
print(f"count: {len(thin)}   complaints: {int(thin['complaints'].sum())} of {int(pay20['complaints'].sum())}")
print(thin[["issue", "complaints", "lenders_any", "ratio", "spread_pp"]]
        .to_string(index=False, float_format=lambda x: f"{x:.2f}"))

=== Issues dominated by small samples (Payday) ===
count: 3   complaints: 34 of 4708
                                                  issue  complaints  lenders_any  ratio  spread_pp
       Unable to get your credit report or credit score           6            6   1.48       0.00
          Problem with fraud alerts or security freezes          13           10   1.42       0.98
Credit monitoring or identity theft protection services          15           12   0.89       1.23


In [23]:
# An Issue is only usable here if enough lenders clear a meaningful share threshold.
d = df[df["Sub-product"] == PAYDAY]
n = d.groupby("canonical_company").size()
rows = []
for key, issue in zip(issues["key"], issues["issue"]):
    share = (d[d["Issue"] == issue].groupby("canonical_company").size() / n).dropna()
    rows.append({"key": key, "issue": issue, "lenders_any": len(share),
                 "lenders_ge5pct": int((share >= 0.05).sum()),
                 "lenders_ge10pct": int((share >= 0.10).sum()),
                 "lenders_ge20pct": int((share >= 0.20).sum())})
usable = pd.DataFrame(rows).sort_values("lenders_ge10pct", ascending=False)
usable.head(12).to_string(index=False)

"key                                                                           issue  lenders_any  lenders_ge5pct  lenders_ge10pct  lenders_ge20pct\ni01                                      Charged fees or interest you didn't expect          220             216              209              171\ni03                                                     Struggling to pay your loan          225             215              197              161\ni10                                            Received a loan you didn't apply for          182             164              132               95\ni12                                                Can't contact lender or servicer          150             122               97               65\ni11                                   Can't stop withdrawals from your bank account          112              93               70               52\ni06                          Problem with the payoff process at the end of the loan          132             10

### The same test across every sub-product

In [24]:
rows = []
for s in sorted(df["Sub-product"].unique()):
    t = differentiate(s, 20)
    if t.empty:
        continue
    top = t.iloc[0]
    rows.append({"Sub-product": s, "lenders>=20": int(top["n_lenders"]),
                 "issues_tested": len(t), "above_null": int(t["above_null"].sum()),
                 "top_issue": top["issue"], "top_ratio": top["ratio"],
                 "top_spread_pp": top["spread_pp"]})
pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:8.2f}")

"                    Sub-product  lenders>=20  issues_tested  above_null                                     top_issue  top_ratio  top_spread_pp\n             Earned wage access            4              5           3                Problems receiving the advance       5.26          31.81\n               Installment loan          149             12          10    Charged fees or interest you didn't expect      14.87          46.27\nOther advances of future income            5              5           2                         Issues with repayment       2.37          30.63\n                      Pawn loan            1              4           0    Charged fees or interest you didn't expect       0.00           0.00\n                    Payday loan           70             15          12 Can't stop withdrawals from your bank account      19.21          29.17\n        Personal line of credit           76             14          12    Charged fees or interest you didn't expect       9.00 

## 7. Response features: spread and degeneracy

In [25]:
big = feat[feat["n_complaints"] >= MIN_N]
rc = ["rate__timely_response", "rate__not_timely", "rate__monetary_relief",
      "rate__nonmonetary_relief", "rate__untimely_response", "rate__in_progress"]
rows = []
for c in rc:
    s = big[c]
    rows.append({"feature": c, "min": s.min(), "p25": s.quantile(.25), "median": s.median(),
                 "p75": s.quantile(.75), "max": s.max(),
                 "at_exactly_0": int((s <= 1e-9).sum()),
                 "at_exactly_1": int((s >= 1 - 1e-9).sum())})
out = pd.DataFrame(rows).set_index("feature").T
print(f"groups with n >= {MIN_N}: {len(big):,}")
out

groups with n >= 20: 316


feature,rate__timely_response,rate__not_timely,rate__monetary_relief,rate__nonmonetary_relief,rate__untimely_response,rate__in_progress
min,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
p25,0.971731,0.000000,0.000000,0.000000,0.0,0.000000
median,1.000000,0.000000,0.000000,0.000000,0.0,0.000000
p75,1.000000,0.028269,0.035244,0.057898,0.0,0.009381
max,1.000000,1.000000,0.523810,0.804598,1.0,0.495798
at_exactly_0,4.000000,189.000000,182.000000,183.000000,294.0,221.000000
at_exactly_1,189.000000,4.000000,0.000000,0.000000,4.0,0.000000


In [26]:
print("=== in-progress right-censoring ===")
ip = raw[raw["Company response to consumer"] == "In progress"]
print(f"  in-progress complaints          : {len(ip):,}")
print(f"  earliest received               : {ip['Date received'].min()}")
print(f"  all on/after 2026-03-18         : {(ip['Date received'] >= '2026-03-18').all()}")
print("\n=== the two 'untimely' definitions ===")
print(f"  flagged 'Untimely response'          : {int((raw[RESP] == 'Untimely response').sum()):,}")
print(f"  Timely response? == 'No'             : {int((raw['Timely response?'] == 'No').sum()):,}")
print(f"  late yet closed with explanation     : {int(((raw['Timely response?'] == 'No') & (raw[RESP] == 'Closed with explanation')).sum()):,}")
print("\n=== sub-issue availability ===")
print(f"  Sub-issue null: {raw['Sub-issue'].isna().mean() * 100:.1f}%  populated in {raw['Sub-issue'].notna().sum():,} rows across {raw.loc[raw['Sub-issue'].notna(),'Issue'].nunique()} Issues")
print("\n=== company public response ===")
pr = raw["Company public response"]
print(f"  null: {pr.isna().sum():,} ({pr.isna().mean() * 100:.1f}%)   categories present: {pr.nunique()}   range {pr.value_counts().min()}-{pr.value_counts().max():,} complaints")

=== in-progress right-censoring ===
  in-progress complaints          : 558
  earliest received               : 2026-03-18T11:30:49.000Z
  all on/after 2026-03-18         : True

=== the two 'untimely' definitions ===
  flagged 'Untimely response'          : 535
  Timely response? == 'No'             : 1,732
  late yet closed with explanation     : 1,157

=== sub-issue availability ===
  Sub-issue null: 88.9%  populated in 4,271 rows across 5 Issues

=== company public response ===
  null: 30,825 (80.3%)   categories present: 9   range 18-5,067 complaints


## 8. Assertions on the claims made below

The conclusions quote specific numbers. This cell recomputes each one from the assembled
table and fails loudly if any drifts, so the narrative cannot silently go stale.

In [27]:
def claim(label, got, want, tol=0.05):
    ok = abs(float(got) - float(want)) <= tol
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<52} got {got:>10.2f}  claimed {want:>10.2f}")
    return ok


res = []
res.append(claim("total groups", len(groups), 2415, 0))
res.append(claim("median group size", groups.median(), 2, 0))
res.append(claim("groups with <5 complaints", int((groups < 5).sum()), 1709, 0))
res.append(claim("groups with <20 complaints", int((groups < 20).sum()), 2099, 0))
for t, g, pc in [(5, 706, 92.8), (10, 458, 88.5), (20, 316, 83.3),
                 (30, 239, 78.5), (50, 163, 70.9), (100, 82, 56.6)]:
    res.append(claim(f"groups >= {t}", int((groups >= t).sum()), g, 0))
    res.append(claim(f"complaint coverage >= {t}", groups[groups >= t].sum() / groups.sum() * 100, pc))

pn = df[df["Sub-product"] == PAYDAY].groupby("canonical_company").size()
res.append(claim("payday groups", len(pn), 482, 0))
for t, g in [(10, 100), (20, 70), (30, 48), (50, 32), (100, 14)]:
    res.append(claim(f"payday lenders >= {t}", int((pn >= t).sum()), g, 0))

get = pay20.set_index("issue")
for issue, r, sp in [("Can't stop withdrawals from your bank account", 19.21, 29.17),
                     ("Charged fees or interest you didn't expect", 11.74, 52.84),
                     ("Money was taken from your bank account on the wrong day or for the wrong amount", 5.50, 14.22),
                     ("Received a loan you didn't apply for", 5.43, 18.47),
                     ("Incorrect information on your report", 5.23, 11.56),
                     ("Struggling to pay your loan", 4.69, 30.85),
                     ("Can't contact lender or servicer", 3.28, 13.12)]:
    res.append(claim(f"ratio: {issue[:34]}", get.loc[issue, "ratio"], r, 0.02))
    res.append(claim(f"spread: {issue[:34]}", get.loc[issue, "spread_pp"], sp, 0.02))
res.append(claim("payday issues tested", len(pay20), 15, 0))
res.append(claim("payday issues above null band", int(pay20["above_null"].sum()), 12, 0))
res.append(claim("max rank movement n>=10 vs n>=20", m["rank_move"].max(), 4, 0))

res.append(claim("timely groups at exactly 1.0", int((big["rate__timely_response"] >= 1 - 1e-9).sum()), 189, 0))
res.append(claim("untimely-flag groups at exactly 0", int((big["rate__untimely_response"] <= 1e-9).sum()), 294, 0))
res.append(claim("monetary groups at exactly 0", int((big["rate__monetary_relief"] <= 1e-9).sum()), 182, 0))
res.append(claim("non-monetary groups at exactly 0", int((big["rate__nonmonetary_relief"] <= 1e-9).sum()), 183, 0))
res.append(claim("in-progress groups at exactly 0", int((big["rate__in_progress"] <= 1e-9).sum()), 221, 0))

res.append(claim("'Untimely response' complaints", int((raw[RESP] == "Untimely response").sum()), 535, 0))
res.append(claim("'Timely response? = No' complaints", int((raw["Timely response?"] == "No").sum()), 1732, 0))
res.append(claim("late but closed with explanation",
                 int(((raw["Timely response?"] == "No") & (raw[RESP] == "Closed with explanation")).sum()), 1157, 0))
res.append(claim("in-progress complaints", len(ip), 558, 0))
res.append(claim("sub-issue null pct", raw["Sub-issue"].isna().mean() * 100, 88.9, 0.1))
res.append(claim("public response null pct", raw["Company public response"].isna().mean() * 100, 80.3, 0.1))
res.append(claim("i18 complaints", int((raw.Issue == "Was approved for a loan, but didn't receive the money").sum()), 258, 0))
res.append(claim("i28 complaints", int((raw.Issue == "Was approved for a loan, but didn't receive money").sum()), 61, 0))

print(f"\n{sum(res)}/{len(res)} claims verified")
assert all(res), "a quoted number in the conclusions does not match the data"

  PASS  total groups                                         got    2415.00  claimed    2415.00
  PASS  median group size                                    got       2.00  claimed       2.00
  PASS  groups with <5 complaints                            got    1709.00  claimed    1709.00
  PASS  groups with <20 complaints                           got    2099.00  claimed    2099.00
  PASS  groups >= 5                                          got     706.00  claimed     706.00
  PASS  complaint coverage >= 5                              got      92.75  claimed      92.80
  PASS  groups >= 10                                         got     458.00  claimed     458.00
  PASS  complaint coverage >= 10                             got      88.47  claimed      88.50
  PASS  groups >= 20                                         got     316.00  claimed     316.00
  PASS  complaint coverage >= 20                             got      83.30  claimed      83.30
  PASS  groups >= 30                    

## Conclusions

### 1. How many lender-product pairs have enough evidence for meaningful comparison?

Across all sub-products there are **2,415** groups, but only **316 (13.1%)** reach 20
complaints and **163 (6.7%)** reach 50. The median group holds **2** complaints, and
**1,709 groups (70.8%)** have fewer than 5.

| threshold | groups | % of groups | % of complaints covered |
| --- | --- | --- | --- |
| >=5 | 706 | 29.2% | 92.8% |
| >=10 | 458 | 19.0% | 88.5% |
| >=20 | 316 | 13.1% | 83.3% |
| >=30 | 239 | 9.9% | 78.5% |
| >=50 | 163 | 6.7% | 70.9% |
| >=100 | 82 | 3.4% | 56.6% |

The gap between group count and complaint coverage is the important part: **13.1% of groups
hold 83.3% of the complaints.** A lender with 8 complaints is a real, named entity in this
table but cannot support a comparison.

For **payday loans** there are 482 groups, of which **100** reach 10, **70** reach 20,
**48** reach 30, **32** reach 50 and **14** reach 100. Coverage is concentrated in three
sub-products: Installment loan (149 groups at >=20), Personal line of credit (76) and
Payday loan (70) account for **295 of the 316**. Title loan (11), Other advances of future
income (5), Earned wage access (4), Pawn loan (1) and Tax refund anticipation loan or check
(0) are not usable at this unit of analysis.

So roughly **70 payday lender-product pairs, and 316 overall, carry enough evidence to
compare**. Everything else is a row in the table but not a finding.

### 2. Which CFPB Issues differentiate payday lenders most?

Ranked by over-dispersion across the 70 payday lenders with >=20 complaints. The metric is
validated by simulation: when every lender is simulated with one shared true rate, the
ratio centres on 1.0 with a 95% null band of at most **[0.65, 1.42]** across base rates from 0.02 to 0.45, so a fixed
decision threshold of **1.45** sits above every simulated band: ratios above it are real
lender heterogeneity, anything at or below it is sampling noise.

| Issue | complaints | lenders | peer share | ratio | p90-p10 (pp) | median share | max share |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Can't stop withdrawals from your bank account | 514 | 52 | 10.9% | **19.21** | 29.2 | 4.2% | 69.7% |
| Charged fees or interest you didn't expect | 1,491 | 68 | 31.7% | **11.74** | 52.8 | 23.6% | 80.9% |
| Money was taken on the wrong day or for the wrong amount | 267 | 47 | 5.7% | 5.50 | 14.2 | 2.0% | 32.3% |
| Received a loan you didn't apply for | 434 | 65 | 9.2% | 5.43 | 18.5 | 8.2% | 63.6% |
| Incorrect information on your report | 137 | 37 | 2.9% | 5.23 | 11.6 | 0.7% | 23.2% |
| Struggling to pay your loan | 809 | 66 | 17.2% | 4.69 | 30.9 | 15.0% | 45.5% |
| Can't contact lender or servicer | 271 | 63 | 5.8% | 3.28 | 13.1 | 5.2% | 28.1% |

**Unauthorized withdrawals differentiate lenders most**, by a factor of 1.6 over the next
Issue. But note its shape: median share is only **4.2%** against a 10.9% peer average, so
the mean is dragged up by a minority of lenders — one reaches 69.7%. This is a
concentrated, lender-specific problem rather than a broad one, and a mean-based feature
would misrepresent it.

**Unexpected fees** has the second-highest ratio (11.74) and by far the largest raw spread
(**52.8 pp**, median 23.6%, max 80.9%). It is the only Issue touching 68 of 70 lenders, so
it is not an artefact of a thin denominator.

`Received a loan you didn't apply for` is notable for a high ratio *and* near-universal
presence (65 of 70 lenders), which makes it the best-behaved of the strong signals.

Only **15 of the 33** Issues occur in payday at all; 12 clear the 1.45 threshold. Three do not:

| Issue | complaints | ratio | reading |
| --- | --- | --- | --- |
| Problem with the payoff process at the end of the loan | 284 | 1.34 | homogeneous across lenders despite high volume |
| Problem with fraud alerts or security freezes | 13 | 1.42 | inside the null band |
| Credit monitoring or identity theft protection services | 15 | 0.89 | indistinguishable from noise |

**Payoff process is the clearest negative result in the data**: 284 complaints across 64
lenders, and those lenders are no more different from each other than chance. It is a large
category that carries almost no lender-discriminating information.

The ranking is **robust to the sample-size threshold**. Recomputed at n>=10 (100 lenders)
the top two are identical and the maximum rank movement anywhere in the table is 4 places,
so nothing here depends on the cut-off.

One caution on interpretation, not on the statistics: `Struggling to pay your loan` has
high raw spread (30.9 pp) but only the 6th-highest ratio, because it is near-universal
(66 of 70 lenders) and its variance is therefore mostly sampling noise around a high base
rate. Its variation is real but weaker than the ranking of the smaller Issues above it.

### 3. Which features appear unsuitable for eventual scoring?

Measured on the 316 groups with >=20 complaints.

**Degenerate — almost no variance to learn from:**

- `rate__timely_response` — median **1.000**, interquartile range **0.972 to 1.000**, and
  **189 of 316** groups sit at exactly 1.0. It ranks almost nobody.
- `rate__untimely_response` — **294 of 316** at exactly 0.
- `rate__monetary_relief` — median 0.000, **182 of 316** at exactly 0 (max 0.52).
- `rate__nonmonetary_relief` — median 0.000, **183 of 316** at exactly 0 (max 0.81).
- `rate__in_progress` — **221 of 316** at exactly 0, and every one of the **558**
  in-progress complaints nationally was received **on or after 2026-03-18**. This is
  right-censoring, not behaviour: those complaints are simply unresolved. Any resolution
  rate over the full window is biased downward, and the bias concentrates in the most
  recent and largest period. This feature is in the table for completeness and should not
  be modelled.

**Conditionally unusable:**

- All 99 issue columns (33 counts + 33 shares + 33 deltas) for the **2,099 groups with <20
  complaints**, where one complaint moves a share by 5 points at n=20 and 50 points at n=2.
- Issue features for the 18 Issues that never appear in payday, and for the 3 thin ones
  holding 34 complaints combined.
- `rate__public_response_null` is carried as a **coverage flag, not a feature**: 80.3% of
  complaints have no public response, and the 9 categories that do exist range from 18 to
  5,067 complaints.
- `Sub-issue` (88.9% null) is not used at all; it is populated in only 4,271 rows and only
  within 5 Issues, three of which are credit-reporting. It would bias any model toward
  credit bureaus.

**Label defects to resolve before production.** The 33 Issue labels are not clean:
`"Was approved for a loan, but didn't receive the money"` (258) and `"Was approved for a
loan, but didn't receive money"` (61) differ only by a trailing comma and match at 0.96 —
one near-duplicate pair, detected automatically, **left unmerged** to preserve the
33-column contract. Meanwhile the superficially similar `"Charged fees or interest you
didn't expect"` (9,225) and `"Unexpected fees"` (173) are *not* duplicates and must never
be merged. Also note unexpected fees is the #1 Issue in only **5 of 8** sub-products; Earned
wage access and Other advances of future income are both dominated by `"Issues with
repayment"`, and Tax refund anticipation by `"Confusing or misleading advertising or
marketing"`.

**Weak but not degenerate:** the credit-reporting Issues. `Incorrect information on your
report` reaches 137 payday complaints and a ratio of 5.23, but only 37 of 70 lenders report
it at all, so much of the remaining sample sits at exactly 0 (median share 0.7%).

### 4. Five to ten candidate features worth investigating

Ordered by strength of evidence above. These are candidates for investigation, not a scoring
set, and no weights are implied.

1. **`dlt__i11` — unauthorized-withdrawal share vs. payday peer baseline.** Highest
   differentiation in the data by a factor of 1.6 over the next Issue. Caveat from the shape
   above: strongly right-skewed, so pair it with a binary/flag version rather than relying
   on the mean.
2. **`shr__i01` and `dlt__i01` — unexpected-fee share, raw and peer-relative.** Largest
   spread (52.8 pp), highest prevalence (68 of 70 lenders), largest volume in the sub-product.
   Keep both: the raw share and the deviation answer different questions, and the
   deviation depends on a peer baseline that the lender itself contributes to.
3. **`dlt__i10` — loan received without applying.** Ratio 5.43, 18.5 pp spread, and 65 of
   70 lenders affected, which makes it the best-conditioned of the strong signals.
4. **`dlt__i17` — money taken on the wrong day or for the wrong amount.** Ratio 5.50, second
   in the table on raw spread among the high-ratio Issues.
5. **`dlt__i12` — cannot contact lender or servicer.** Ratio 3.28, 13.1 pp spread, 63 of 70
   lenders. Weaker than the top four but stable and widespread.
6. **`rate__not_timely` in place of `rate__timely_response`.** Identical information, but
   the complement spans 0 to 1 instead of being crushed against 1.0, and it captures the
   **1,157** complaints that were late yet still closed as "Closed with explanation" — which
   the explicit `Untimely response` flag alone would miss at **535**.
7. **`rate__monetary_relief`.** The outcome closest to the consumer being made whole, and
   the only response feature with genuine spread (0 to 0.52). Requires an n-threshold and
   almost certainly a tail-only or bucketed design given 182 of 316 groups at zero.
8. **`n_complaints`, strictly as a confidence weight and never as a quality signal.** It is
   the most predictive column in the table and the most dangerous one. Listed explicitly so
   that mean 33.0 versus median 2.0 complaints per lender cannot leak into a model as a
   quality proxy.
9. **Explicitly *not* recommended: `dlt__i06` (payoff process).** 284 complaints, ratio 1.34,
   inside the null band. A large category with no lender-discriminating power — a good
   demonstration that complaint volume alone does not imply signal.
10. **Explicitly *not* recommended: `rate__in_progress`.** It measures when we looked, not
    what the lender did. See the censoring note in section 3.

Every feature above remains a **complaint-mix or complaint-process** measure. None is a
rate, because the dataset still has no exposure denominator: nothing here supports a claim
about how often a lender does any of this relative to how much it lends.